<a href="https://colab.research.google.com/github/deeedaniel/gpt/blob/main/buildgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

torch.manual_seed(1337)

cuda


In [ ]:
# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# urllib.request.urlretrieve(url, "input.txt")

dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)

subset = []
for i, example in enumerate(dataset):
    subset.append(example['text'])
    if i >= 50000:
        break
text = "\n".join(subset)
print(len(text))

# with open("input.txt", "r") as f:
#     text = f.read()

# print(len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size, chars)

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

247231797
4522 ['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '\x7f', '\x80', '\x81', '\x82', '\x83', '\x8b', '\x8c', '\x90', '\x91', '\x92', '\x93', '\x94', '\x95', '\x96', '\x97', '\x98', '\x99', '\x9c', '\x9d', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '«', '¬', '\xad', '®', '¯', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '»', '¼', '½', '¾', '¿', 'À', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Æ', 'Ç', 'È', 'É', 'Ì', 'Í', 'Î', 'Ï', 'Ð', 'Ñ', 'Ò', 'Ó', 'Ô', 'Õ', 'Ö', '×', 'Ø', 'Ú', 'Û', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é

In [ ]:
!pip install tokenizers -q
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# Save the new dataset text to input.txt so the tokenizer can train on it
with open("input.txt", "w", encoding="utf-8") as f:
    f.write(text)

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"], vocab_size=1000)
tokenizer.train(files=["input.txt"], trainer=trainer)
tokenizer.save("openwebtext-bpe.json")

vocab_size = tokenizer.get_vocab_size()

def encode(s):
    return tokenizer.encode(s).ids

def decode(l):
    return tokenizer.decode(l)

In [ ]:
# Convert our text into data (tensor)
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:20])

torch.Size([100404019]) torch.int64
tensor([ 52, 353,  17, 508,  17,  52,  86, 834,  16, 330,  69, 226,  77, 352,
         39,  50,  50,  13, 628,  17])


In [ ]:
# Split data into training data and validating data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

90363617 10040402


In [ ]:
block_size = 1024  # Massively increased context window
batch_size = 16    # Reduced to 16 to fit safely within A100 40GB VRAM

def get_batch(split):
  data = train_data if split == 'train' else val_data

  # pick random starting indices for our batches
  ix = torch.randint(len(data) - block_size, (batch_size,))

  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+1+block_size] for i in ix])

  x, y = x.to(device), y.to(device)
  return x,y

In [ ]:
xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb[0])
print(yb[0])
print(decode(xb[0].tolist()))
print(decode(yb[0].tolist()))

torch.Size([16, 1024]) torch.Size([16, 1024])
tensor([575, 339, 253,  ..., 235, 869, 238], device='cuda:0')
tensor([339, 253, 213,  ..., 869, 238,  18], device='cuda:0')
 question a thousand times over. Tabletop gaming is loads of fun, but the sad and simple truth is that, for many people, it’s just not easy to consistently get a group together to play. Chalk it up to busyness, distance, or what have you, sometimes it’s just easier to play games with one person.

This is especially true if you have a partner. Many a couple have turned to board games in their quest to find new ways to bond, and it certainly can be a challenge finding that one game that will satisfy your needs. I’ve played some games in my time, and there has been some hits and misses. I’ve compiled some of my favorite two player games into the following list, a list that I hope can help others find their perfect two players game.

NOTE: Keep in mind that this list is “Zach’s favorite board games for two,” not “the best 

In [ ]:
class BigramModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx)

    loss = None
    if targets is not None:
      B,T,C = logits.shape

      # Flatten batch and time to compare logits and targets
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

model = BigramModel(vocab_size).to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([16384, 1000])
tensor(7.3986, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
# --- OLD BIGRAM MODEL CODE ---
# Commented out so it doesn't accidentally overwrite our new GPT hyper-parameters (like batch_size)

# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
#
# batch_size = 4 # <--- This was secretly overriding your batch_size!
#
# for step in range(5000):
#   xb,yb = get_batch('train')
#   logits, loss = model(xb,yb)
#   optimizer.zero_grad(set_to_none=True)
#   loss.backward()
#   optimizer.step()
#
#   if step % 500 == 0:
#     print(f"step {step}: loss {loss.item():.4f}")
#
# print("final loss: ", loss.item())

In [ ]:
def generate(model, idx, max_new_tokens):
  for _ in range(max_new_tokens):
    idx_con = idx[:, -block_size:]
    logits, loss = model(idx_con)
    logits = logits[:, -1, :]
    probs = F.softmax(logits, dim=-1)
    idx_next = torch.multinomial(probs, num_samples=1)
    idx = torch.cat((idx, idx_next), dim=1)
  return idx

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)  # start with a single "character 0" as a seed
generated = generate(model, context, max_new_tokens=50)
print(decode(generated[0].tolist()))

ldreat thenitedumatesicM ne�ook peousEday butplictread_ had�” point), reg blublicollroup� gu 19�^ratherobweasedD It imp decoss&ud rec; ob


In [ ]:
embed_dim = 768
head_size = 64

class Head(nn.Module):
  def __init__(self, embed_dim, head_size, block_size):
    super().__init__()
    self.key = nn.Linear(embed_dim, head_size, bias=False, device=device)
    self.query = nn.Linear(embed_dim, head_size, bias=False, device=device)
    self.value = nn.Linear(embed_dim, head_size, bias=False, device=device)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)
    v = self.value(x)

    # FlashAttention: Highly optimized, automatically handles the causal mask (is_causal=True)
    out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return out

In [ ]:
token_embedding_table = nn.Embedding(vocab_size, embed_dim, device=device)
x = token_embedding_table(xb)   # xb from before, shape (32... wait, actually (4,8) if you kept batch_size=4 for xb, or reflect your current batch_size)
print(x.shape)

head = Head(embed_dim, head_size, block_size).to(device)
out = head(x)
print(out.shape)

torch.Size([16, 1024, 768])
torch.Size([16, 1024, 64])


In [ ]:
num_heads = 12

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, embed_dim, head_size, block_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(embed_dim, head_size, block_size) for _ in range(num_heads)])
    self.proj = nn.Linear(num_heads * head_size, embed_dim)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    return out

In [ ]:
mha = MultiHeadAttention(num_heads, embed_dim, head_size, block_size).to(device)
out = mha(x)
print(out.shape)

torch.Size([16, 1024, 768])


In [ ]:
class FeedForward(nn.Module):
  def __init__(self, embed_dim):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(embed_dim, 4 * embed_dim),
        nn.ReLU(),
        nn.Linear(4*embed_dim, embed_dim),
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, embed_dim, num_heads, block_size):
    super().__init__()
    head_size = embed_dim // num_heads
    self.sa = MultiHeadAttention(num_heads, embed_dim, head_size, block_size)
    self.ffwd = FeedForward(embed_dim)
    self.ln1 = nn.LayerNorm(embed_dim)
    self.ln2 = nn.LayerNorm(embed_dim)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

In [ ]:
block = Block(embed_dim, num_heads, block_size).to(device)
out = block(x)
print(out.shape)

torch.Size([16, 1024, 768])


In [ ]:
num_layers = 12

class TinyGPT(nn.Module):
  def __init__(self, vocab_size, embed_dim, block_size, num_heads, num_layers):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embed_dim, device=device)
    self.position_embedding_table = nn.Embedding(block_size, embed_dim, device=device)
    self.blocks = nn.Sequential(*[Block(embed_dim, num_heads, block_size) for _ in range(num_layers)]).to(device)
    self.ln_f = nn.LayerNorm(embed_dim).to(device)
    self.lm_head = nn.Linear(embed_dim, vocab_size).to(device)

  def forward(self, idx, targets=None):
    B,T = idx.shape
    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device))
    x = tok_emb + pos_emb
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)

    loss = None
    if targets is not None:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

In [ ]:
import gc
gc.collect() # Force Python to delete unreferenced objects
torch.cuda.empty_cache() # Clear the VRAM from the OOM crash

model = TinyGPT(vocab_size, embed_dim, block_size, num_heads, num_layers).to(device)

# Wrap the test forward pass in no_grad() AND autocast to save massive amounts of memory!
with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(xb, yb)

print(logits.shape)
print(loss)

torch.Size([16384, 1000])
tensor(7.0499, device='cuda:0')


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

for step in range(10000):
    xb, yb = get_batch('train')
    optimizer.zero_grad(set_to_none=True)

    # Autocast to bfloat16 for massively accelerated A100 training
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(xb, yb)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step}: loss {loss.item():.4f}")

print("final loss:", loss.item())

step 0: loss 7.0499
step 100: loss 4.7944
step 200: loss 4.4801
step 300: loss 4.2943
step 400: loss 4.3190
step 500: loss 4.2491
step 600: loss 4.1941
step 700: loss 4.2813
step 800: loss 4.0869
step 900: loss 3.9387
step 1000: loss 3.8356
step 1100: loss 3.7526
step 1200: loss 3.5709
step 1300: loss 3.5630
step 1400: loss 3.3370
step 1500: loss 3.2778
step 1600: loss 3.1075
step 1700: loss 3.1894
step 1800: loss 3.1342
step 1900: loss 2.9789
step 2000: loss 2.9890
step 2100: loss 2.9351
step 2200: loss 2.8879
step 2300: loss 2.8582
step 2400: loss 3.0465
step 2500: loss 2.9879
step 2600: loss 2.8749
step 2700: loss 2.8335
step 2800: loss 2.7487
step 2900: loss 2.8436
step 3000: loss 2.7395
step 3100: loss 2.8291
step 3200: loss 2.7946
step 3300: loss 2.7729
step 3400: loss 2.7016
step 3500: loss 2.7514
step 3600: loss 2.7906
step 3700: loss 2.8521
step 3800: loss 2.7625
step 3900: loss 2.6501
step 4000: loss 2.5149
step 4100: loss 2.7212
step 4200: loss 2.8034
step 4300: loss 2.5170


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = generate(model, context, max_new_tokens=1000)
print(decode(generated[0].tolist()))

 Children [Cardin Burns] was a mysterious nature works rank interrogating of the UF-78 with Thomas Ellis’s coach, whilst have been playing Mane Get? While this post does something else make these enormous faces combat the quantities of their work here.

At his behavior as the reason his obsession did had millenness, and then Burns would emerge as he had accurately played – vire Get himself.

However, if that mania was so, it might not be seemed to be – that it would be a little place now if any idea would be a super earned cocker like an inhaled jerk winter or a trash-o-mash winter.

But would the time do a super ears? That was the difference only maken inhaling in front of one – that in a concert over his “manic” concert experienced a sexual behavior and the direction of which he no longer treated or insign into the offense would have...

Are there? Just for a 19th defeated, worked at a college camp in a greater aspiring for the death of his art favor coming back, add and publicity, a

## Fine-Tuning
To fine-tune the model on a new dataset, we load the new text, encode it using our **existing tokenizer** (so the vocabulary mapping stays exactly the same), and train the existing model with a much smaller learning rate.

In [ ]:
# 1. Load your fine-tuning dataset (e.g., a specific custom text)
# dataset_ft = load_dataset("your_dataset", split="train")
# text_ft = "\n".join([ex['text'] for ex in dataset_ft])

# For demonstration, a placeholder string
code_dataset = load_dataset("sahil2801/CodeAlpaca-20k")

# 2. Encode using the ALREADY TRAINED tokenizer
data_ft = torch.tensor(encode(text_ft), dtype=torch.long)

# Split into train/val
n_ft = int(0.9 * len(data_ft))
train_data_ft = data_ft[:n_ft]
val_data_ft = data_ft[n_ft:]

def get_batch_ft(split):
    data = train_data_ft if split == 'train' else val_data_ft
    # Make sure we don't pick an index out of bounds if the FT dataset is small
    max_idx = max(1, len(data) - block_size)
    ix = torch.randint(max_idx, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

# 3. Fine-tune with a smaller learning rate (e.g., 1e-5 instead of 5e-4)
optimizer_ft = torch.optim.AdamW(model.parameters(), lr=1e-5)

print("Starting fine-tuning...")
for step in range(500): # Shorter fine-tuning run
    xb, yb = get_batch_ft('train')
    optimizer_ft.zero_grad(set_to_none=True)

    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(xb, yb)

    loss.backward()
    optimizer_ft.step()

    if step % 100 == 0:
        print(f"FT step {step}: loss {loss.item():.4f}")

Starting fine-tuning...
FT step 0: loss 0.3159
FT step 100: loss 0.0055
FT step 200: loss 0.0010
FT step 300: loss 0.0004
FT step 400: loss 0.0003
